# Use Case 4: Agentic Workflows (Task Queues)

**The Concept:** 
When multiple AI agents collaborate (e.g., a "Researcher Agent" and a "Writer Agent"), they need a reliable way to hand off asynchronous tasks to each other without dropping them if an agent crashes.

**The Architecture:** 
We use SochDB's decentralized **Priority Queues**. Agents can enqueue tasks for each other. If an agent crashes while generating a report, its "Visibility Timeout" expires, and the task is safely returned to the queue for another agent to pick up.

---

### Step 0: Install Packages & Connect LLM Backend
Our Writer Agent will be powered by Google Gemini. We ensure credentials are secure and properly loaded via Python dotenv.

In [ ]:
!pip install sochdb openai python-dotenv

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

CHAT_MODEL = "gemini-3-flash-preview"
EMBEDDING_MODEL = "gemini-embedding-001"


### Step 1: Initialize the Priority Queue
Agents connect to the shared decentralized SQLite-like database.

In [ ]:
from sochdb import Database, PriorityQueue

db = Database.open("./agent_workflow_db")

# Connect to the shared task queue bus
queue = PriorityQueue.from_database(db, "agent_task_bus")

### Step 2: The Researcher Agent delegates a task
The upstream agent finishes its job and enqueues the next sub-task for downstream agents.

In [ ]:
import json

# The Researcher Agent formats its event logic as JSON and enqueues to the SochDB Queue
payload = json.dumps({"task": "write_summary", "topic": "Quantum Computing Architecture"}).encode('utf-8')

queue.enqueue(priority=100, payload=payload)
print("Researcher Agent: Task delegated to the Writer Agents! (📤)")

### Step 3: The Writer Agent claims and processes the task
The downstream agent pulls the task, processes the payload using the real Google Gemini model, and successfully acknowledges (`ack()`) the completion to permanently remove it from the system.

In [ ]:
# The Writer Agent pulls the highest priority task available
task = queue.dequeue(worker_id="writer_agent_alpha")

if task:
    task_data = json.loads(task.payload.decode('utf-8'))
    print(f"Writer Agent: Claimed task for topic -> '{task_data['topic']}'")
    
    try:
        print("Writer Agent: Generating report via Gemini Gemini-3-Flash...")
        # Execute real Agent Action
        final_report = generate_report(task_data['topic'])
        print(f"\n---\n[Agent Output]\n{final_report}\n---\n")
        
        # Acknowledge success to permanently remove it from the queue
        queue.ack(task.task_id)
        print("Writer Agent: Report finished and task safely acknowledged (✅).")
        
    except Exception as e:
        # If the LLM call fails, NACK returns it to the queue for another agent
        queue.nack(task.task_id)
        print(f"Writer Agent: Failed... returning task to queue (❌). Error: {str(e)}")
else:
    print("Writer Agent: No pending tasks in the queue.")